# ACDOCA Synthetic Generator — BigQuery Studio Notebook

Interactive UI for the synthetic ACDOCA generator, runnable inside [BigQuery Studio](https://cloud.google.com/bigquery/docs/notebooks-introduction). Pick parameters with widgets, click **Generate**, output goes to a BigQuery table.

**Prerequisites** (per `docs/superpowers/specs/...` plan, Phase 0):
- GCP project with BigQuery + Cloud Storage APIs enabled
- A BigQuery dataset (e.g. `synthetic_acdoca`)
- A GCS bucket for the Spark BigQuery connector's staging files
- IAM: `roles/bigquery.dataEditor`, `roles/bigquery.jobUser`, `roles/dataproc.editor` on the project, plus `objectAdmin` on the GCS bucket

**Runtime**: pick the **PySpark** runtime when BQ Studio prompts. The Spark BigQuery connector and Dataproc Serverless backing are auto-attached.

**Repo source**: this notebook clones `https://github.com/GDAVIDREEVES/bigquery-sap-synthetic-data.git` and installs the `acdoca_generator` package in editable mode. **Push your local commits first** (`git push origin main`) so the notebook gets the latest code (non-goods IC flows, year-end trueup, controversy tags).

## 1. Setup — clone repo + install package

Run this cell once per kernel restart. If the clone target already exists, the cell skips re-cloning.

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/GDAVIDREEVES/bigquery-sap-synthetic-data.git"
REPO_DIR = "/home/jupyter/repo"

if not os.path.exists(REPO_DIR):
    subprocess.check_call(["git", "clone", REPO_URL, REPO_DIR])
else:
    subprocess.check_call(["git", "-C", REPO_DIR, "pull", "--ff-only"])

# Editable install so future `git pull` updates take effect on kernel restart
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR])

# ipywidgets is preinstalled on most BQ Studio runtimes; install if missing
try:
    import ipywidgets  # noqa: F401
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ipywidgets"])

print("Repo and package installed. Kernel restart not required for the imports below.")

## 2. Configure BigQuery target + GCS staging bucket

**Edit these three values for your GCP project** before running the form. The values become defaults inside the widgets so each Generate click writes to the same place.

In [ ]:
# === EDIT FOR YOUR PROJECT ===
PROJECT_ID = "acdoca-synthetic-greg"   # gcloud project id
BQ_DATASET = "synthetic_acdoca"        # created via notebooks/00_bq_setup.sql
BQ_TABLE   = "journal_entries"         # connector creates on first write
GCS_BUCKET = "acdoca-synthetic-greg-bq-staging"  # connector temp staging
# =============================

FULL_BQ_TABLE = f"{PROJECT_ID}.{BQ_DATASET}.{BQ_TABLE}"
os.environ["ACDOCA_BQ_TABLE"] = FULL_BQ_TABLE
os.environ["ACDOCA_GCS_TEMP_BUCKET"] = GCS_BUCKET

# BQ Studio Spark runtime auto-attaches the BigQuery connector — no need to set
# spark.jars.packages or ACDOCA_SPARK_BQ_PACKAGE.

print(f"Target: {FULL_BQ_TABLE}")
print(f"GCS staging: gs://{GCS_BUCKET}")

## 3. Imports + Spark session

BQ Studio supplies the SparkSession. `getOrCreate()` reuses it.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from acdoca_generator.generators.pipeline import GenerationConfig, generate_acdoca_dataframe
from acdoca_generator.utils.spark_writer import GenerationParams, write_acdoca_table
from acdoca_generator.config.industries import industry_keys
from acdoca_generator.config.presets import DEMO_PRESETS, get_preset, preset_keys

spark = SparkSession.builder.getOrCreate()
print("Spark version:", spark.version)

## 4. Interactive form

Pick a preset to seed the form, override any field, click **Generate**. The form runs the same code path as `scripts/run_generate_bq.py` and writes to the BigQuery table you configured above.

Changing the **Preset** dropdown auto-fills the other widgets from `DEMO_PRESETS[...]`. Pick `custom` to disable auto-fill and tune everything manually.

In [ ]:
import ipywidgets as W
from IPython.display import display, clear_output

_SUPPORTED_ISOS = [
    "US", "DE", "GB", "FR", "IE", "CH", "IN", "CN", "JP",
    "BR", "MX", "NL", "SG", "KR", "AU", "IT", "ES", "BE",
    "DK", "CA", "IL", "PL", "PH", "CR", "ID", "RU",
]

preset_dd = W.Dropdown(
    options=["custom"] + preset_keys(),
    value="quick_smoke",
    description="Preset",
)
industry_dd = W.Dropdown(
    options=industry_keys(),
    value="pharmaceutical",
    description="Industry",
)
country_mc = W.SelectMultiple(
    options=_SUPPORTED_ISOS,
    value=("US", "DE", "CH", "IE"),
    description="Countries",
    rows=8,
)
year_slider = W.IntSlider(value=2026, min=2020, max=2030, description="Year")
fiscal_variant_rb = W.RadioButtons(
    options=["calendar", "april"],
    value="calendar",
    description="Fiscal",
)
complexity_rb = W.RadioButtons(
    options=["light", "medium", "high", "very_high"],
    value="medium",
    description="Tier",
)
txn_slider = W.IntSlider(
    value=200, min=50, max=2000, step=50,
    description="Txn/CC/period",
    style={"description_width": "initial"},
)
ic_pct_slider = W.FloatSlider(
    value=0.30, min=0.0, max=0.6, step=0.05,
    description="IC %",
    readout_format=".2f",
)
use_industry_ic = W.Checkbox(
    value=False,
    description="Use industry-default IC % (overrides slider)",
    style={"description_width": "initial"},
)
include_reversals = W.Checkbox(value=True, description="Reversals (~5%)")
include_closing   = W.Checkbox(value=True, description="Closing entries")
include_sc        = W.Checkbox(value=True, description="Supply chain")
sc_chains_slider  = W.IntSlider(
    value=50, min=1, max=500, step=10,
    description="SC chains",
)
include_segment_pl = W.Checkbox(value=True, description="Segment P&L")
include_trueup     = W.Checkbox(value=True, description="Year-end trueup")
challenged_slider  = W.FloatSlider(
    value=0.0, min=0.0, max=0.5, step=0.05,
    description="Challenged %",
    readout_format=".2f",
    style={"description_width": "initial"},
)
seed_int = W.IntText(value=42, description="Seed")

go_btn = W.Button(description="Generate", button_style="primary", icon="play")
out = W.Output(layout={"border": "1px solid #ddd", "padding": "6px", "min_height": "120px"})

_ALL_INPUTS = [
    industry_dd, country_mc, year_slider, fiscal_variant_rb,
    complexity_rb, txn_slider, ic_pct_slider, use_industry_ic,
    include_reversals, include_closing, include_sc, sc_chains_slider,
    include_segment_pl, include_trueup, challenged_slider, seed_int,
]


def _apply_preset(_change):
    """When the preset dropdown changes, fill the rest of the form."""
    key = preset_dd.value
    if key == "custom":
        return
    p = get_preset(key)
    industry_dd.value = p.industry_key
    isos = tuple(c.strip() for c in p.country_isos_csv.split(",") if c.strip())
    # Filter to supported list to avoid widget-value-not-in-options errors
    country_mc.value = tuple(i for i in isos if i in _SUPPORTED_ISOS)
    year_slider.value = p.fiscal_year
    fiscal_variant_rb.value = p.fiscal_variant
    complexity_rb.value = p.complexity
    txn_slider.value = min(max(p.txn_per_cc_per_period, txn_slider.min), txn_slider.max)
    if p.ic_pct is None:
        use_industry_ic.value = True
    else:
        use_industry_ic.value = False
        ic_pct_slider.value = p.ic_pct
    include_reversals.value = p.include_reversals
    include_closing.value = p.include_closing
    include_sc.value = p.include_supply_chain
    sc_chains_slider.value = p.sc_chains_per_period
    include_segment_pl.value = p.include_segment_pl
    challenged_slider.value = float(getattr(p, "challenged_share", 0.0))


preset_dd.observe(_apply_preset, names="value")
_apply_preset(None)  # seed initial widget state from quick_smoke


def _on_generate(_btn):
    with out:
        clear_output()
        # Disable all inputs while running
        for w in _ALL_INPUTS + [preset_dd, go_btn]:
            w.disabled = True
        try:
            country_isos = list(country_mc.value)
            if not country_isos:
                print("Pick at least one country.")
                return
            cfg = GenerationConfig(
                industry_key=industry_dd.value,
                country_isos=country_isos,
                fiscal_year=int(year_slider.value),
                fiscal_variant=fiscal_variant_rb.value,
                complexity=complexity_rb.value,
                txn_per_cc_per_period=int(txn_slider.value),
                include_reversals=bool(include_reversals.value),
                include_closing=bool(include_closing.value),
                seed=int(seed_int.value),
                ic_pct=None if use_industry_ic.value else float(ic_pct_slider.value),
                include_supply_chain=bool(include_sc.value),
                sc_chains_per_period=int(sc_chains_slider.value),
                include_segment_pl=bool(include_segment_pl.value),
                include_year_end_trueup=bool(include_trueup.value),
                challenged_share=float(challenged_slider.value),
            )
            print(f"Generating: industry={cfg.industry_key} countries={country_isos} "
                  f"year={cfg.fiscal_year} complexity={cfg.complexity} "
                  f"txn/cc/period={cfg.txn_per_cc_per_period} seed={cfg.seed}")
            print("Spark generation may take 30-90s on first run (Dataproc Serverless cold start) ...")
            result = generate_acdoca_dataframe(spark, cfg)
            print("Generation complete. Writing to BigQuery ...")
            gen_params = GenerationParams(
                industry=cfg.industry_key,
                complexity=cfg.complexity,
                countries_iso_csv=",".join(country_isos),
                fiscal_year=cfg.fiscal_year,
                seed=cfg.seed,
                validation_profile="strict",
            )
            write_acdoca_table(
                spark, result.acdoca_df,
                full_table_name=os.environ["ACDOCA_BQ_TABLE"],
                gen=gen_params,
                output_format="bigquery",
                gcs_temp_bucket=os.environ["ACDOCA_GCS_TEMP_BUCKET"],
            )
            n_rows = result.acdoca_df.count()
            print(f"✓ Wrote {n_rows:,} rows to {os.environ['ACDOCA_BQ_TABLE']}")
            if result.supply_chain_flows_df is not None:
                n_sc = result.supply_chain_flows_df.count()
                print(f"  + {n_sc:,} supply-chain flow rows (in-memory only; not written to BQ)")
        except Exception as e:
            print(f"FAILED: {type(e).__name__}: {e}")
            raise
        finally:
            for w in _ALL_INPUTS + [preset_dd, go_btn]:
                w.disabled = False


go_btn.on_click(_on_generate)

form = W.VBox([
    W.HTML("<h3>ACDOCA generator parameters</h3>"),
    preset_dd,
    W.HBox([industry_dd, year_slider, fiscal_variant_rb]),
    country_mc,
    W.HBox([complexity_rb, txn_slider]),
    W.HBox([ic_pct_slider, use_industry_ic]),
    W.HBox([include_reversals, include_closing]),
    W.HBox([include_sc, sc_chains_slider]),
    W.HBox([include_segment_pl, include_trueup, challenged_slider]),
    seed_int,
    go_btn,
    W.HTML("<h4>Output</h4>"),
    out,
])
display(form)

## 5. Verify the write

After clicking Generate, this SQL cell counts rows + samples one. BQ Studio supports a SQL cell type — change the cell to **SQL** via the cell-type dropdown if you want syntax highlighting, or leave it as Python and run `spark.sql(...)` style as below.

In [ ]:
from google.cloud import bigquery

client = bigquery.Client(project=PROJECT_ID)
df = client.query(f"""
    SELECT COUNT(*) AS rows,
           MIN(GJAHR) AS y_min,
           MAX(GJAHR) AS y_max,
           COUNT(DISTINCT RBUKRS) AS n_company_codes
    FROM `{FULL_BQ_TABLE}`
""").to_dataframe()
df

In [ ]:
# Sample 5 rows for sanity check
client.query(f"SELECT * FROM `{FULL_BQ_TABLE}` LIMIT 5").to_dataframe()

## 6. Operational TP slices (optional)

These queries exercise the new modules from commit `bd61e51`: non-goods IC flows, year-end trueup, controversy/APA tagging.

### Flow type distribution (AWREF prefix)
Confirms goods + royalty + service + cost-share + trueup all emit.

In [ ]:
client.query(f"""
    SELECT SUBSTR(AWREF, 1, 2) AS prefix, COUNT(*) AS rows
    FROM `{FULL_BQ_TABLE}`
    WHERE AWREF IS NOT NULL AND AWREF != ''
    GROUP BY prefix
    ORDER BY prefix
""").to_dataframe()

In [ ]:
# Year-end trueup rows (only land if include_year_end_trueup=True AND an LRD's margin
# was outside the target band)
client.query(f"""
    SELECT RBUKRS, RACCT, WSL, BLART, POPER, AWREF, SGTXT
    FROM `{FULL_BQ_TABLE}`
    WHERE STARTS_WITH(AWREF, 'TU')
    ORDER BY AWREF, BUZEI
    LIMIT 20
""").to_dataframe()